# 🌿 Plant Disease Detector — Exploration & Visualisation

Ce notebook explore les données, entraîne les modèles et visualise les résultats.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')
print('✅ Imports OK')

## 1. Génération et exploration des données agronomiques

In [ ]:
from src.ml.train import generate_synthetic_csv
df = generate_synthetic_csv('../data/raw/agronomic_data.csv', n_samples=1000)
print(df.shape)
df.head()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
features = ['temperature', 'humidity', 'rainfall', 'soil_ph', 'nitrogen',
            'phosphorus', 'potassium', 'leaf_area_index', 'plant_age_days', 'previous_infection']

colors = ['#22c55e', '#f97316', '#374151', '#a78bfa', '#ef4444']
classes = df['label'].unique()

for ax, feat in zip(axes.flatten(), features):
    for i, cls in enumerate(classes):
        subset = df[df['label'] == cls][feat]
        ax.hist(subset, alpha=0.5, bins=20, color=colors[i], label=cls)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')

axes[0, 0].legend(fontsize=7)
plt.suptitle('Distribution des features par classe', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Matrice de corrélation

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Matrice de corrélation des features', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Entraînement ML

In [ ]:
from src.ml.train import train as train_ml
metrics = train_ml(csv_path='../data/raw/agronomic_data.csv', model_out='../models/ml_model.pkl')
print('Metrics:', metrics)

## 4. Entraînement DL (CNN)

In [ ]:
from src.dl.train import train as train_dl
history = train_dl(data_dir='../data/raw', model_out='../models/cnn_model.pt', epochs=5, batch_size=16)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train', color='#22c55e')
ax1.plot(history['val_loss'], label='Val', color='#f97316')
ax1.set_title('Loss'); ax1.legend()

ax2.plot(history['val_acc'], color='#58a6ff')
ax2.set_title('Val Accuracy')
plt.tight_layout(); plt.show()

## 5. Test de la fusion DL + ML

In [ ]:
from src.ml.predict import predict_tabular
from src.dl.predict import _dummy_prediction
from src.utils.fusion import fuse_predictions

features = {
    'temperature': 28.0, 'humidity': 78.0, 'rainfall': 85.0,
    'soil_ph': 6.2, 'nitrogen': 45.0, 'phosphorus': 22.0,
    'potassium': 28.0, 'leaf_area_index': 3.5,
    'plant_age_days': 120, 'previous_infection': 1,
}

ml_result = predict_tabular(features, model_path='../models/ml_model.pkl')
dl_result  = _dummy_prediction()  # Remplacer par predict_image() avec une vraie image

final = fuse_predictions(dl_result, ml_result)

print(f"{final['emoji']} Diagnostic : {final['final_class']}")
print(f"   Confiance  : {final['confidence']:.2%}")
print(f"   Sévérité   : {final['severity']}")
print(f"   Traitement : {final['treatment']}")